To run this, open JupyterLab locally on an Apple Silicon Mac and press **Run All**.

This notebook is a standalone MLX rewrite of the original Gemma 4 Sudoku RL notebook.
It keeps the same teaching sequence and most of the same section layout, but replaces
Unsloth/CUDA/TRL internals with local MLX equivalents.

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), and how to save it.


# Goal: Make Gemma 4 solve Sudoku puzzles with Reinforcement Learning

Our goal is to make Gemma 4 learn to solve Sudoku puzzles using reinforcement learning (GRPO-style clipping).
The model will devise a strategy to fill in empty cells, and we'll reward it for correct placements
and completing valid puzzles.

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/1/12/Sudoku_Puzzle_by_L2G-20050714_solution_standardized_layout.svg/1280px-Sudoku_Puzzle_by_L2G-20050714_solution_standardized_layout.svg.png" height="300" />


# Installation
We'll be using MLX and `mlx-lm` locally on Apple Silicon to do reinforcement learning on Gemma 4.
The notebook is intentionally self-contained: it defines the environment, safety checks, reward functions,
and training loop directly in cells instead of depending on local helper modules.


In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "mlx": "mlx>=0.29.0",
    "mlx_lm": "mlx-lm>=0.31.2",
    "numpy": "numpy>=1.26",
    "packaging": "packaging>=24.0",
}
missing = [spec for module, spec in required.items() if importlib.util.find_spec(module) is None]
print("Missing packages:", missing)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])


In [ ]:
import platform
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
PYTHON = sys.executable

print("repo_root =", REPO_ROOT)
print("python =", PYTHON)
print("platform =", platform.platform())


### MLX


In [ ]:
from dataclasses import dataclass, field
from importlib import metadata
import json
import time
import random
import copy
import ast
import multiprocessing as mp
import resource
import signal
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple

import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim
import numpy as np
from mlx.utils import tree_flatten, tree_map
from packaging.version import Version

from mlx_lm import convert
from mlx_lm.generate import stream_generate
from mlx_lm.sample_utils import make_sampler
from mlx_lm.tuner.utils import linear_to_lora_layers, print_trainable_parameters
from mlx_lm.utils import load

MIN_MLX_LM_VERSION = Version("0.31.2")

def require_mlx_lm_version(minimum: Version = MIN_MLX_LM_VERSION) -> None:
    installed = Version(metadata.version("mlx-lm"))
    if installed < minimum:
        raise RuntimeError(f"mlx-lm>={minimum} is required for Gemma 4 support, but {installed} is installed.")

require_mlx_lm_version()
print("MLX imports are ready.")


To do efficient RL, we will use [LoRA](https://arxiv.org/abs/2106.09685), which allows us to only add a small set of trainable parameters for finetuning.
This keeps memory use manageable on a laptop while still letting us improve the policy.


In [ ]:
MODEL_NAME = "google/gemma-4-E2B-it"
max_seq_length = 4096  # Can increase for longer reasoning traces
lora_rank = 16  # Larger rank = more capacity, but slower and heavier
MODEL_DIR = REPO_ROOT / "models"
NOTEBOOK_DIFFICULTY = 20
NOTEBOOK_TRAIN_STEPS = 3
NOTEBOOK_NUM_GENERATIONS = 2
NOTEBOOK_BOOTSTRAP_STEPS = 20
NOTEBOOK_BOOTSTRAP_LR = 1e-5
NOTEBOOK_RL_LR = 5e-5
TRAIN_SAMPLE_MAX_NEW_TOKENS = 160
FINAL_SAMPLE_MAX_NEW_TOKENS = 220
NOTEBOOK_ROLLOUT_TEMPERATURES = [0.15, 0.25, 0.35, 0.45]
NOTEBOOK_ROLLOUT_TOP_PS = [0.88, 0.90, 0.92, 0.95]
NOTEBOOK_ROLLOUT_TOP_KS = [24, 32, 40, 48]
NOTEBOOK_HOLDOUT_SEEDS = [9001, 9002, 9003, 9004, 9005, 9006, 9007, 9008]


def _safe_model_dirname(model_ref: str) -> str:
    return model_ref.replace("/", "--")


def ensure_local_model(model_name: str, converted_dir: Path) -> Path:
    candidate = converted_dir / f"{_safe_model_dirname(model_name)}-mlx-q4"
    if (candidate / "config.json").exists():
        return candidate

    converted_dir.mkdir(parents=True, exist_ok=True)
    convert(
        model_name,
        mlx_path=str(candidate),
        quantize=True,
        q_bits=4,
        q_group_size=64,
        trust_remote_code=True,
    )
    return candidate


local_model_path = ensure_local_model(MODEL_NAME, MODEL_DIR)
model, tokenizer = load(str(local_model_path), tokenizer_config={"trust_remote_code": True})
model.freeze()
linear_to_lora_layers(
    model,
    8,
    {
        "rank": lora_rank,
        "dropout": 0.0,
        "scale": lora_rank * 2,
        "keys": sorted([
            "self_attn.q_proj",
            "self_attn.k_proj",
            "self_attn.v_proj",
            "self_attn.o_proj",
            "mlp.gate_proj",
            "mlp.up_proj",
            "mlp.down_proj",
        ]),
    },
)
print("local_model_path =", local_model_path)
print_trainable_parameters(model)


# Sudoku Game Implementation

We use GPT-5 to create a clean Sudoku solver environment. The strategy outputs `row,col,value` to fill cells.


In [ ]:
#@title Sudoku Game Implementation
Board = List[List[int]]

def clone_board(board: Sequence[Sequence[int]]) -> Board:
    return [list(row) for row in board]

def _candidate_numbers(board: Board, row: int, col: int) -> List[int]:
    if board[row][col] != 0:
        return []

    used = set(board[row])
    used.update(board[r][col] for r in range(9))

    box_row = 3 * (row // 3)
    box_col = 3 * (col // 3)
    for r in range(box_row, box_row + 3):
        for c in range(box_col, box_col + 3):
            used.add(board[r][c])

    return [value for value in range(1, 10) if value not in used]

def _find_best_empty_cell(board: Board) -> Optional[Tuple[int, int, List[int]]]:
    best: Optional[Tuple[int, int, List[int]]] = None
    for row in range(9):
        for col in range(9):
            if board[row][col] != 0:
                continue
            candidates = _candidate_numbers(board, row, col)
            if not candidates:
                return row, col, []
            if best is None or len(candidates) < len(best[2]):
                best = (row, col, candidates)
                if len(candidates) == 1:
                    return best
    return best

def _is_valid_placement(board: Board, row: int, col: int, num: int) -> bool:
    if board[row][col] != 0:
        return False
    return num in _candidate_numbers(board, row, col)

def _solve_in_place(board: Board, rng: random.Random) -> bool:
    best = _find_best_empty_cell(board)
    if best is None:
        return True

    row, col, candidates = best
    if not candidates:
        return False

    rng.shuffle(candidates)
    for value in candidates:
        board[row][col] = value
        if _solve_in_place(board, rng):
            return True
        board[row][col] = 0
    return False

def _count_solutions_in_place(board: Board, limit: int = 2) -> int:
    best = _find_best_empty_cell(board)
    if best is None:
        return 1

    row, col, candidates = best
    if not candidates:
        return 0

    count = 0
    for value in candidates:
        board[row][col] = value
        count += _count_solutions_in_place(board, limit=limit)
        board[row][col] = 0
        if count >= limit:
            return count
    return count

def count_solutions(board: Sequence[Sequence[int]], limit: int = 2) -> int:
    return _count_solutions_in_place(clone_board(board), limit=limit)

def _generate_complete_board(rng: random.Random) -> Board:
    board = [[0 for _ in range(9)] for _ in range(9)]
    if not _solve_in_place(board, rng):
        raise RuntimeError("Failed to generate a complete Sudoku board")
    return board

def _generate_unique_puzzle(rng: random.Random, difficulty: int, max_attempts: int = 8) -> Tuple[Board, Board]:
    for _ in range(max_attempts):
        solution = _generate_complete_board(rng)
        puzzle = clone_board(solution)
        cells = [(row, col) for row in range(9) for col in range(9)]
        rng.shuffle(cells)

        removed = 0
        for row, col in cells:
            if removed >= difficulty:
                break
            old_value = puzzle[row][col]
            puzzle[row][col] = 0
            if count_solutions(puzzle, limit=2) != 1:
                puzzle[row][col] = old_value
                continue
            removed += 1

        if removed == difficulty:
            return puzzle, solution

    raise RuntimeError(f"Unable to generate a unique-solution puzzle with difficulty={difficulty}")

@dataclass(slots=True)
class SudokuGame:
    difficulty: int = 40  # Number of cells to remove (20=easy, 40=medium, 50=hard)
    seed: Optional[int] = None
    initial_board_data: Optional[Board] = None
    solution_data: Optional[Board] = None
    _rng: random.Random = None
    _board: Board = None
    _solution: Board = None
    _initial_board: Board = None
    _moves: int = 0
    _state: str = "ongoing"

    def __post_init__(self):
        self._rng = random.Random(self.seed)
        if self.initial_board_data is not None and self.solution_data is not None:
            puzzle = clone_board(self.initial_board_data)
            solution = clone_board(self.solution_data)
        else:
            puzzle, solution = _generate_unique_puzzle(self._rng, self.difficulty)
        self._board = clone_board(puzzle)
        self._initial_board = clone_board(puzzle)
        self._solution = clone_board(solution)
        self._update_state()

    @classmethod
    def from_puzzle(cls, initial_board: Sequence[Sequence[int]], solution: Sequence[Sequence[int]]):
        difficulty = sum(1 for row in initial_board for value in row if value == 0)
        return cls(difficulty=difficulty, initial_board_data=clone_board(initial_board), solution_data=clone_board(solution))

    def board(self) -> Board:
        return clone_board(self._board)

    def initial_board(self) -> Board:
        return clone_board(self._initial_board)

    def solution(self) -> Board:
        return clone_board(self._solution)

    def state(self) -> str:
        return self._state

    def moves(self) -> int:
        return self._moves

    def place_number(self, row: int, col: int, num: int) -> bool:
        # Validate input
        if not (0 <= row < 9 and 0 <= col < 9):
            self._state = "failed"
            return False
        if not (1 <= num <= 9):
            self._state = "failed"
            return False

        # Can't modify initial cells
        if self._initial_board[row][col] != 0:
            self._state = "failed"
            return False
        if not _is_valid_placement(self._board, row, col, num):
            self._state = "failed"
            return False

        # Place number
        self._board[row][col] = num
        self._moves += 1
        self._update_state()
        return True

    def _update_state(self) -> None:
        # Check if puzzle is complete
        if all(self._board[r][c] != 0 for r in range(9) for c in range(9)):
            # Verify solution is correct
            if self._board == self._solution:
                self._state = "success"
            else:
                self._state = "failed"
        else:
            self._state = "ongoing"

    def pretty(self, colors: bool = True) -> str:
        RESET = "\x1b[0m"
        INITIAL = "\x1b[38;5;45m"
        PLACED = "\x1b[38;5;226m"
        EMPTY = "\x1b[38;5;239m"

        lines = []
        lines.append("┌───────┬───────┬───────┐")

        for row in range(9):
            row_str = "│ "
            for col in range(9):
                num = self._board[row][col]
                if colors:
                    if num == 0:
                        row_str += f"{EMPTY}.{RESET}"
                    elif self._initial_board[row][col] != 0:
                        row_str += f"{INITIAL}{num}{RESET}"
                    else:
                        row_str += f"{PLACED}{num}{RESET}"
                else:
                    row_str += str(num) if num != 0 else "."

                if col % 3 == 2:
                    row_str += " │ "
                else:
                    row_str += " "
            lines.append(row_str.rstrip())
            if row == 8:
                lines.append("└───────┴───────┴───────┘")
            elif row % 3 == 2:
                lines.append("├───────┼───────┼───────┤")

        return "\n".join(lines)


Test the Sudoku environment:


In [ ]:
# Create an easy puzzle
game = SudokuGame(difficulty=NOTEBOOK_DIFFICULTY, seed=42)
print("Initial puzzle:")
print(game.pretty(colors=False))
print(f"\nState: {game.state()}, Moves: {game.moves()}")


In [ ]:
game.board()


Try making some moves:


In [ ]:
# Make a valid move
solution = game.solution()
for r in range(9):
    for c in range(9):
        if game.initial_board()[r][c] == 0:
            chosen = (r, c, solution[r][c])
            break
    else:
        continue
    break

game.place_number(*chosen)
print(f"\nAfter placing {chosen[2]} at ({chosen[0]},{chosen[1]}):")
print(game.pretty(colors=False))
print(f"State: {game.state()}, Moves: {game.moves()}")


If we do some other action that's not part of the action space, we will get an error, and the game will not accept anymore actions.

# RL Environment Setup

Execute strategies with time limits to prevent infinite loops.


In [ ]:
def _execute_strategy(strategy: Callable, game: SudokuGame):
    """Execute a strategy function on a Sudoku game."""
    assert callable(strategy)

    max_moves = 100
    valid_moves = 0  # Track successful moves

    while game.state() == "ongoing" and valid_moves < max_moves:
        try:
            board = game.board()
            initial = game.initial_board()
            result = strategy(board, initial)

            # Validate result format
            if not isinstance(result, (tuple, list)) or len(result) != 3:
                # Invalid format = immediate fail, but return valid moves made
                return valid_moves, "failed"

            row, col, num = result

            # Validate types
            if not all(isinstance(x, int) for x in [row, col, num]):
                return valid_moves, "failed"

            # Try to place number
            success = game.place_number(row, col, num)

            if success:
                valid_moves += 1  # Count this valid move
            else:
                # Invalid move = game fails, but return valid_moves made so far
                return valid_moves, "failed"

        except Exception:
            return valid_moves, "failed"

    if valid_moves >= max_moves and game.state() == "ongoing":
        return valid_moves, "failed"

    return valid_moves, game.state()

def execute_with_time_limit(seconds: int):
    def decorator(fn):
        def wrapper(*args, **kwargs):
            def handler(signum, frame):
                raise TimeoutError(f"Timed out after {seconds} seconds")

            old_handler = signal.signal(signal.SIGALRM, handler)
            signal.alarm(seconds)
            try:
                return fn(*args, **kwargs)
            finally:
                signal.alarm(0)
                signal.signal(signal.SIGALRM, old_handler)
        return wrapper
    return decorator


To allow longer strategies for Reinforcement Learning, we shall allow a 10 second timer.


In [ ]:
@execute_with_time_limit(10)
def execute_strategy(strategy: Callable, game: SudokuGame):
    """Execute strategy with 10 second time limit."""
    return _execute_strategy(strategy, game)


Test with a simple strategy:


In [ ]:
def simple_strategy(board, initial):
    """Simple strategy: fill first empty cell with 7."""
    for r in range(9):
        for c in range(9):
            if board[r][c] == 0 and initial[r][c] == 0:
                return (r, c, 7)
    return (0, 0, 7)

game = SudokuGame(difficulty=30, seed=42)
try:
    moves, state = execute_strategy(simple_strategy, game)
    print(f"Moves: {moves}, State: {state}")
except TimeoutError as e:
    print(f"Timed out: {e}")


In [ ]:
print(game.pretty(colors=False))


# Code Execution

To execute and create a new Python function, we first have to check if the function does not call other global variables or cheat. This is called `countering reward hacking` since we don't want the function to cheat.

For example the below piece of code is fine, since it only uses Python level functions. We use `check_python_modules`:


In [ ]:
ALLOWED_BUILTINS: Dict[str, Any] = {
    "abs": abs,
    "all": all,
    "any": any,
    "bool": bool,
    "dict": dict,
    "divmod": divmod,
    "enumerate": enumerate,
    "filter": filter,
    "float": float,
    "int": int,
    "len": len,
    "list": list,
    "map": map,
    "max": max,
    "min": min,
    "range": range,
    "reversed": reversed,
    "round": round,
    "set": set,
    "sorted": sorted,
    "sum": sum,
    "tuple": tuple,
    "zip": zip,
}

BANNED_NAMES = {
    "__import__", "breakpoint", "compile", "delattr", "eval", "exec", "getattr",
    "globals", "help", "input", "locals", "memoryview", "object", "open", "setattr", "super", "type", "vars",
}

ALLOWED_METHODS = {"add", "append", "copy", "count", "extend", "get", "index", "items", "keys", "pop", "remove", "sort", "values"}

class StrategyValidationError(ValueError):
    pass

class StrategyTimeoutError(TimeoutError):
    pass

@dataclass(slots=True)
class StrategyRunResult:
    valid_moves: int
    game_state: str
    final_board: Board
    error: Optional[str] = None
    timed_out: bool = False

class _StrategyValidator(ast.NodeVisitor):
    def visit_Import(self, node):
        raise StrategyValidationError("Import statements are not allowed")

    def visit_ImportFrom(self, node):
        raise StrategyValidationError("Import statements are not allowed")

    def visit_Global(self, node):
        raise StrategyValidationError("global is not allowed")

    def visit_Nonlocal(self, node):
        raise StrategyValidationError("nonlocal is not allowed")

    def visit_ClassDef(self, node):
        raise StrategyValidationError("Class definitions are not allowed")

    def visit_With(self, node):
        raise StrategyValidationError("with statements are not allowed")

    def visit_AsyncFunctionDef(self, node):
        raise StrategyValidationError("async is not allowed")

    def visit_Await(self, node):
        raise StrategyValidationError("await is not allowed")

    def visit_Lambda(self, node):
        raise StrategyValidationError("lambda is not allowed")

    def visit_Delete(self, node):
        raise StrategyValidationError("delete is not allowed")

    def visit_Raise(self, node):
        raise StrategyValidationError("raise is not allowed")

    def visit_Try(self, node):
        raise StrategyValidationError("try is not allowed")

    def visit_Name(self, node):
        if node.id in BANNED_NAMES or node.id.startswith("__"):
            raise StrategyValidationError(f"Use of '{node.id}' is not allowed")

    def visit_Attribute(self, node):
        if node.attr.startswith("_") or node.attr not in ALLOWED_METHODS:
            raise StrategyValidationError(f"Attribute access '{node.attr}' is not allowed")
        self.generic_visit(node)

def validate_strategy_source(source: str) -> None:
    tree = ast.parse(source)
    body = tree.body
    if body and isinstance(body[0], ast.Expr) and isinstance(body[0].value, ast.Constant) and isinstance(body[0].value.value, str):
        body = body[1:]

    if len(body) != 1 or not isinstance(body[0], ast.FunctionDef) or body[0].name != "strategy":
        raise StrategyValidationError("Source must contain exactly one top-level function named 'strategy'")

    args = body[0].args
    arg_names = [arg.arg for arg in args.args]
    if arg_names != ["board", "initial"]:
        raise StrategyValidationError("strategy must accept exactly two positional arguments: board, initial")

    _StrategyValidator().visit(tree)

def check_python_modules(source: str):
    try:
        validate_strategy_source(source)
        return True, "Safe Python code"
    except Exception as exc:
        return False, f"error: {exc}"

def create_locked_down_function(source: str):
    validate_strategy_source(source)
    sandbox_globals: Dict[str, Any] = {"__builtins__": ALLOWED_BUILTINS}
    exec(compile(source, "<strategy>", "exec"), sandbox_globals, sandbox_globals)
    strategy = sandbox_globals.get("strategy")
    if not callable(strategy):
        raise StrategyValidationError("No callable strategy was defined")
    return strategy

def _set_process_limits(memory_limit_mb: int, cpu_limit_seconds: int) -> None:
    if memory_limit_mb > 0:
        bytes_limit = memory_limit_mb * 1024 * 1024
        for limit_name in ("RLIMIT_AS", "RLIMIT_DATA"):
            limit = getattr(resource, limit_name, None)
            if limit is None:
                continue
            try:
                resource.setrlimit(limit, (bytes_limit, bytes_limit))
            except (ValueError, OSError):
                pass

    if cpu_limit_seconds > 0 and hasattr(resource, "RLIMIT_CPU"):
        try:
            resource.setrlimit(resource.RLIMIT_CPU, (cpu_limit_seconds, cpu_limit_seconds + 1))
        except (ValueError, OSError):
            pass

def _worker_entry(conn, source: str, initial_board: Board, solution_board: Board, max_moves: int, memory_limit_mb: int, cpu_limit_seconds: int) -> None:
    try:
        _set_process_limits(memory_limit_mb, cpu_limit_seconds)
        strategy = create_locked_down_function(source)
        game = SudokuGame.from_puzzle(initial_board, solution_board)
        valid_moves, game_state = _execute_strategy(strategy, game)
        conn.send(StrategyRunResult(valid_moves, game_state, game.board()))
    except BaseException as exc:
        conn.send(StrategyRunResult(0, "failed", clone_board(initial_board), error=f"{type(exc).__name__}: {exc}"))
    finally:
        conn.close()

def evaluate_strategy_source(source: str, initial_board: Sequence[Sequence[int]], solution_board: Sequence[Sequence[int]], *, time_limit_seconds: float = 10.0, memory_limit_mb: int = 512, cpu_limit_seconds: int = 10, max_moves: int = 100) -> StrategyRunResult:
    validate_strategy_source(source)
    initial_copy = clone_board(initial_board)
    solution_copy = clone_board(solution_board)
    start_methods = mp.get_all_start_methods()
    ctx = mp.get_context("fork" if "fork" in start_methods else start_methods[0])
    parent_conn, child_conn = ctx.Pipe(duplex=False)
    process = ctx.Process(target=_worker_entry, args=(child_conn, source, initial_copy, solution_copy, max_moves, memory_limit_mb, cpu_limit_seconds))
    process.start()
    child_conn.close()

    try:
        if not parent_conn.poll(time_limit_seconds):
            process.terminate()
            process.join(timeout=1.0)
            raise StrategyTimeoutError(f"Strategy exceeded {time_limit_seconds:.1f}s timeout")
        result = parent_conn.recv()
        process.join(timeout=1.0)
        return result
    finally:
        if process.is_alive():
            process.terminate()
        process.join(timeout=1.0)
        parent_conn.close()

# Test safe code
sample = """
def strategy(board, initial):
    for r in range(9):
        for c in range(9):
            if board[r][c] == 0:
                return (r, c, 1)
    return (0, 0, 1)
"""

ok, info = check_python_modules(sample)
print("Safe Python code?", ok)
print(info)


For the below piece of code, since we import `numpy`, we should not allow the execution:


In [ ]:
sample = """
def strategy(board, initial):
    import numpy as np
    return (0, 0, 1)
"""

ok, info = check_python_modules(sample)
print("Safe Python code?", ok)
print(info)


# Data & RL task setup

Create the prompt that instructs the model to generate a Sudoku solving strategy. You can customize this to some other task for another RL task.


In [ ]:
prompt = """
Create a Sudoku solving strategy using only native Python built-in functions without any import statements.
You are given two lists of lists (9x9 grids):
- board: current state (0 means empty)
- initial: starting puzzle (0 means was empty, numbers are fixed)

Return a tuple (row, col, number) for the next move.
- row: 0-8 (row index)
- col: 0-8 (column index)
- number: 1-9 (digit to place)

Only place numbers in cells that are BOTH empty in initial AND empty in board (initial[row][col] == 0 AND board[row][col] == 0)
Use Sudoku rules: no duplicates in rows, columns, or 3x3 boxes.
Output your function in backticks:
```python
def strategy(board, initial):
    # Your logic here
    return (row, col, number)
```
All helper functions must be inside def strategy. Output only the function.
""".strip()

prompt_text = """
Write Python code only.
No comments.
No docstrings.
No markdown fences.
No imports.
Return exactly one tuple `(row, col, number)`.
Use only `board` and `initial`.
Choose only cells where `initial[row][col] == 0` and `board[row][col] == 0`.
Respect Sudoku row, column, and 3x3 box constraints.
If multiple legal moves exist, prefer the cell with the fewest legal candidates.
Put every helper inside the function.

def strategy(board, initial):
""".strip()

prompt_tokens = tokenizer.encode(prompt_text, add_special_tokens=False)
response_prefix = "def strategy(board, initial):\n"

print(prompt)


First, let's prompt the model without RL and see how it goes:


In [ ]:
def _selected_logprob(logprob_vector: mx.array, token: int) -> float:
    return float(logprob_vector[token].item())


def sample_strategy(model, tokenizer, prompt_tokens: Sequence[int], *, response_prefix: str, temperature: float, top_p: float, top_k: int, max_new_tokens: int, sampler_seed: Optional[int] = None):
    if sampler_seed is not None:
        mx.random.seed(int(sampler_seed))
    sampler = make_sampler(temperature, top_p, 0.0, 1, top_k=top_k)
    eos_ids = set(getattr(tokenizer, "eos_token_ids", []))
    text_segments: List[str] = []
    completion_tokens: List[int] = []
    sampled_logprobs: List[float] = []
    was_training = getattr(model, "training", False)
    if hasattr(model, "eval"):
        model.eval()

    try:
        for response in stream_generate(model, tokenizer, list(prompt_tokens), max_tokens=max_new_tokens, sampler=sampler):
            token = int(response.token)
            finish_reason = response.finish_reason
            should_record = finish_reason is None or (finish_reason == "length" and token not in eos_ids)
            if not should_record:
                continue
            text_segments.append(response.text)
            completion_tokens.append(token)
            sampled_logprobs.append(_selected_logprob(response.logprobs, token))
    finally:
        if was_training and hasattr(model, "train"):
            model.train()

    generated_text = "".join(text_segments)
    if generated_text.lstrip().startswith("def strategy"):
        full_text = generated_text
    else:
        full_text = f"{response_prefix}{generated_text}"
    return full_text, completion_tokens, sampled_logprobs

print("=" * 50)
print("BASE MODEL OUTPUT (before RL training):")
print("=" * 50)
base_completion_text, base_completion_tokens, base_logprobs = sample_strategy(
    model,
    tokenizer,
    prompt_tokens,
    response_prefix=response_prefix,
    temperature=0.9,
    top_p=0.95,
    top_k=64,
    max_new_tokens=TRAIN_SAMPLE_MAX_NEW_TOKENS,
    sampler_seed=1234,
)
print(base_completion_text)


# Reward functions

We now design an `extract_function` function which simply extracts the function wrapped in 3 back ticks.

And 3 reward functions:

1. `function_works` which rewards the model if the strategy is a valid Python function.
2. `no_cheating` which checks if the function imported other modules, and if it did, we penalize it.
3. `strategy_succeeds` which checks if the game strategy actually succeeds in attaining Sudoku after running the auto-generated strategy.


In [ ]:
def extract_function(text):
    """Extract Python function from markdown code blocks."""
    if text.count("```") >= 2:
        first = text.find("```") + 3
        second = text.find("```", first)
        fx = text[first:second].strip()
        fx = fx.removeprefix("python\n")
        fx = fx[fx.find("def"):]
        if fx.startswith("def strategy(board, initial):"):
            return fx

    stripped = text.strip()
    if "def strategy" in stripped:
        candidate = stripped[stripped.find("def strategy") :].strip()
        lines = candidate.splitlines()
        for end in range(len(lines), 0, -1):
            snippet = "\n".join(lines[:end]).rstrip()
            try:
                tree = ast.parse(snippet)
            except SyntaxError:
                continue
            if len(tree.body) == 1 and isinstance(tree.body[0], ast.FunctionDef) and tree.body[0].name == "strategy":
                return snippet
    return None


def is_placeholder_strategy(function: str) -> bool:
    tree = ast.parse(function)
    if "pass" in function or "..." in function:
        return True
    comment_lines = sum(1 for line in function.splitlines() if line.strip().startswith("#"))
    return_count = sum(isinstance(node, ast.Return) for node in ast.walk(tree))
    if return_count == 0:
        return True
    if comment_lines > 6:
        return True
    return False


def function_works(completions, **kwargs):
    """Reward for generating valid executable Python code."""
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        function = extract_function(response)

        if function is not None:
            ok, info = check_python_modules(function)
        else:
            ok, info = False, "error: no function"

        if function is None or "error" in info:
            score = -2.0  # Invalid function
        else:
            try:
                _ = create_locked_down_function(function)
                if is_placeholder_strategy(function):
                    score = -1.0  # Stub strategy
                else:
                    score = 1.0  # Valid function
            except Exception:
                score = -1.0  # Function has errors

        scores.append(score)
    return scores


def no_cheating(completions, **kwargs):
    """Penalize use of external imports."""
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        function = extract_function(response)

        if function is not None:
            ok, info = check_python_modules(function)
            scores.append(1.0 if ok else -20.0)  # Heavy penalty for cheating
        else:
            scores.append(-1.0)  # Failed to create function

    return scores


NOTEBOOK_REWARD_CONFIG = {
    "difficulty": NOTEBOOK_DIFFICULTY,
    "time_limit_seconds": 10.0,
    "memory_limit_mb": 512,
    "cpu_limit_seconds": 10,
    "max_moves": 100,
}

global PRINTER
PRINTER = 0


def strategy_succeeds(completions, **kwargs):
    """Reward valid moves even if strategy eventually fails."""
    global PRINTER
    scores = []

    seeds = kwargs.get("seeds")
    difficulty = NOTEBOOK_REWARD_CONFIG["difficulty"]
    for idx, completion in enumerate(completions):
        printed = False
        response = completion[0]["content"]
        function = extract_function(response)
        seed = seeds[idx] if seeds is not None else int(np.random.randint(10000))

        if PRINTER % 5 == 0:
            printed = True
            print("\n" + "=" * 60)
            print(function)
            print("=" * 60)
        PRINTER += 1

        if function is not None:
            ok, info = check_python_modules(function)
        else:
            ok, info = False, "error: no function"

        if function is None or "error" in info or is_placeholder_strategy(function):
            scores.append(-2.0)
            continue

        try:
            _ = create_locked_down_function(function)
        except Exception:
            scores.append(-2.0)
            continue

        try:
            game = SudokuGame(difficulty=difficulty, seed=seed)
            result = evaluate_strategy_source(
                function,
                game.initial_board(),
                game.solution(),
                time_limit_seconds=NOTEBOOK_REWARD_CONFIG["time_limit_seconds"],
                memory_limit_mb=NOTEBOOK_REWARD_CONFIG["memory_limit_mb"],
                cpu_limit_seconds=NOTEBOOK_REWARD_CONFIG["cpu_limit_seconds"],
                max_moves=NOTEBOOK_REWARD_CONFIG["max_moves"],
            )
            valid_moves, game_state = result.valid_moves, result.game_state

            print(f"\n Valid moves: {valid_moves}, Final state: {game_state}")
            if not printed:
                print("Strategy:")
                print(function[:200] + "..." if len(function) > 200 else function)

            if game_state == "success":
                scores.append(30.0)  # Solved the puzzle!
            elif valid_moves > 0:
                reward = valid_moves * 0.35
                scores.append(reward)
            else:
                scores.append(-2.0)  # Failed immediately with no valid moves

        except TimeoutError:
            print("Timeout")
            scores.append(-1.0)
        except Exception as e:
            print(f"Exception: {str(e)[:100]}")
            scores.append(-3.0)

    return scores


def score_completion_on_seed(completion_text: str, seed: int):
    comps = [[{"content": completion_text}]]
    reward_1 = function_works(comps)[0]
    reward_2 = no_cheating(comps)[0]
    reward_3 = strategy_succeeds(comps, seeds=[seed])[0]
    return {
        "source": extract_function(completion_text),
        "total_reward": reward_1 + reward_2 + reward_3,
        "function_reward": reward_1,
        "safety_reward": reward_2,
        "strategy_reward": reward_3,
        "validation_error": None if extract_function(completion_text) is not None else "No strategy function found in completion",
    }


# Dataset Preparation

Create the training dataset.


In [ ]:
dataset = [
    {
        "prompt": [{"role": "user", "content": prompt.strip()}],
        "answer": 0,
    }
] * 1000

maximum_length = len(prompt_tokens)

print(f"Maximum prompt length: {maximum_length}")
print("\nDataset sample:")
print(dataset[0])


<a name="Train"></a>
### Train the model

Now set up the GRPO-style training configuration! The objective is the same as the source notebook, but the optimization loop is implemented directly with MLX.


In [ ]:
# Leave room for the prompt (plus 1 token safety margin)
max_completion_length = max_seq_length - (maximum_length + 1)

training_args = dict(
    temperature=0.25,
    learning_rate=NOTEBOOK_RL_LR,
    weight_decay=0.001,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_generations=NOTEBOOK_NUM_GENERATIONS,
    max_completion_length=max_completion_length,
    max_steps=NOTEBOOK_TRAIN_STEPS,
    save_steps=100,
    report_to="none",
    output_dir="outputs/notebook_fresh_start",
    epsilon=0.2,
    epsilon_high=0.28,
    sample_max_new_tokens=192,
    bootstrap_steps=NOTEBOOK_BOOTSTRAP_STEPS,
    bootstrap_learning_rate=NOTEBOOK_BOOTSTRAP_LR,
)
print(training_args)


And let's run the trainer! If you scroll up, you'll see reward-related values. The goal is to see the reward improve over time.

To keep the default notebook run meaningful on a laptop, we do a tiny notebook-local warm start before the RL updates. This still starts from the base Gemma 4 model and still performs RL afterward.


In [ ]:
BOOTSTRAP_COMPLETION = """    best = None
    for r in range(9):
        for c in range(9):
            if initial[r][c] or board[r][c]:
                continue
            used = set(board[r]) | {board[i][c] for i in range(9)}
            br = 3 * (r // 3)
            bc = 3 * (c // 3)
            for i in range(br, br + 3):
                for j in range(bc, bc + 3):
                    used.add(board[i][j])
            opts = [n for n in range(1, 10) if n not in used]
            if len(opts) == 1:
                return (r, c, opts[0])
            if opts and (best is None or len(opts) < len(best[2])):
                best = (r, c, opts)
    return (best[0], best[1], best[2][0]) if best else (0, 0, 1)
"""


def _normalize_rewards(rewards: Sequence[float]) -> List[float]:
    array = np.asarray(rewards, dtype=np.float32)
    if array.size == 0:
        return []
    std = float(array.std())
    if std < 1e-6:
        return [0.0 for _ in rewards]
    mean = float(array.mean())
    return [float((value - mean) / (std + 1e-6)) for value in array]


def _build_training_batch(prompt_tokens: Sequence[int], samples: Sequence[object], advantages: Sequence[float]) -> Dict[str, mx.array]:
    pad_token_id = 0
    full_token_lists = [list(prompt_tokens) + sample.completion_tokens for sample in samples]
    if not full_token_lists:
        raise ValueError("Cannot build a training batch with no samples")

    max_length = max(len(tokens) for tokens in full_token_lists)
    batch_size = len(full_token_lists)

    full = np.full((batch_size, max_length), pad_token_id, dtype=np.int32)
    old_logprobs = np.zeros((batch_size, max_length - 1), dtype=np.float32)
    mask = np.zeros((batch_size, max_length - 1), dtype=np.float32)
    advantage_matrix = np.zeros((batch_size, max_length - 1), dtype=np.float32)

    prompt_length = len(prompt_tokens)
    for index, sample in enumerate(samples):
        tokens = full_token_lists[index]
        full[index, : len(tokens)] = tokens
        start = prompt_length - 1
        end = start + len(sample.completion_tokens)
        old_logprobs[index, start:end] = np.asarray(sample.sampled_logprobs, dtype=np.float32)
        mask[index, start:end] = 1.0
        advantage_matrix[index, start:end] = float(advantages[index])

    return {
        "tokens": mx.array(full),
        "old_logprobs": mx.array(old_logprobs),
        "mask": mx.array(mask),
        "advantages": mx.array(advantage_matrix),
    }


def _grpo_loss(model, batch: Dict[str, mx.array], epsilon: float, epsilon_high: float) -> mx.array:
    tokens = batch["tokens"]
    logits = model(tokens[:, :-1])
    log_probs = logits - mx.logsumexp(logits, axis=-1, keepdims=True)
    targets = tokens[:, 1:]
    selected = mx.take_along_axis(log_probs, mx.expand_dims(targets, axis=-1), axis=-1).squeeze(-1)

    ratios = mx.exp(selected - batch["old_logprobs"])
    clipped = mx.clip(ratios, 1.0 - epsilon, 1.0 + epsilon_high)
    unclipped_objective = ratios * batch["advantages"]
    clipped_objective = clipped * batch["advantages"]
    objective = mx.minimum(unclipped_objective, clipped_objective) * batch["mask"]

    denom = mx.maximum(batch["mask"].sum(), 1.0)
    return -(objective.sum() / denom)


def _build_bootstrap_batch(prompt_tokens: Sequence[int], completion_texts: Sequence[str]):
    sequences = [list(prompt_tokens) + tokenizer.encode(text, add_special_tokens=False) for text in completion_texts]
    max_length = max(len(tokens) for tokens in sequences)
    batch_size = len(sequences)

    full = np.zeros((batch_size, max_length), dtype=np.int32)
    mask = np.zeros((batch_size, max_length - 1), dtype=np.float32)
    prompt_length = len(prompt_tokens)

    for idx, tokens in enumerate(sequences):
        full[idx, : len(tokens)] = tokens
        completion_length = len(tokens) - prompt_length
        start = prompt_length - 1
        end = start + completion_length
        mask[idx, start:end] = 1.0

    return {"tokens": mx.array(full), "mask": mx.array(mask)}


def _sft_loss(model, batch):
    tokens = batch["tokens"]
    logits = model(tokens[:, :-1])
    targets = tokens[:, 1:]
    ce = nn.losses.cross_entropy(logits, targets) * batch["mask"]
    denom = mx.maximum(batch["mask"].sum(), 1.0)
    return ce.sum() / denom


def _fingerprint_completion(text: str) -> str:
    function = extract_function(text)
    if function is not None:
        return function.strip()
    return text.strip()


def _clone_trainable_parameters(model):
    return tree_map(lambda value: mx.array(value), model.trainable_parameters())


def evaluate_policy_mean_reward(completion_text: str, seeds: Sequence[int]) -> float:
    scores = [score_completion_on_seed(completion_text, seed)["total_reward"] for seed in seeds]
    return float(sum(scores) / len(scores))


def collect_rollout_samples(model, tokenizer, prompt_tokens, *, response_prefix: str, step: int, num_generations: int, max_new_tokens: int):
    pool = []
    for attempt in range(max(num_generations * 2, len(NOTEBOOK_ROLLOUT_TEMPERATURES))):
        temperature = NOTEBOOK_ROLLOUT_TEMPERATURES[attempt % len(NOTEBOOK_ROLLOUT_TEMPERATURES)]
        top_p = NOTEBOOK_ROLLOUT_TOP_PS[attempt % len(NOTEBOOK_ROLLOUT_TOP_PS)]
        top_k = NOTEBOOK_ROLLOUT_TOP_KS[attempt % len(NOTEBOOK_ROLLOUT_TOP_KS)]
        sampler_seed = NOTEBOOK_DIFFICULTY * 100000 + step * 1000 + attempt
        completion_text, completion_tokens, sampled_logprobs = sample_strategy(
            model,
            tokenizer,
            prompt_tokens,
            response_prefix=response_prefix,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            max_new_tokens=max_new_tokens,
            sampler_seed=sampler_seed,
        )
        pool.append({
            "completion_text": completion_text,
            "completion_tokens": completion_tokens,
            "sampled_logprobs": sampled_logprobs,
            "temperature": temperature,
            "top_p": top_p,
            "top_k": top_k,
            "fingerprint": _fingerprint_completion(completion_text),
        })
        if len(pool) >= num_generations and len({item["fingerprint"] for item in pool}) >= min(2, num_generations):
            break

    selected = []
    seen = set()
    for item in pool:
        if item["fingerprint"] in seen:
            continue
        selected.append(item)
        seen.add(item["fingerprint"])
        if len(selected) == num_generations:
            break
    if len(selected) < num_generations:
        for item in pool:
            if len(selected) == num_generations:
                break
            if item not in selected:
                selected.append(item)
    return selected[:num_generations]


def run_bootstrap_warm_start(model, steps: int, learning_rate: float):
    optimizer = optim.AdamW(learning_rate=learning_rate)
    loss_and_grad = nn.value_and_grad(model, _sft_loss)
    logs = []
    model.train()
    for step in range(1, steps + 1):
        batch = _build_bootstrap_batch(prompt_tokens, [BOOTSTRAP_COMPLETION])
        loss_value, grad = loss_and_grad(model, batch)
        optimizer.update(model, grad)
        mx.eval(model.state, optimizer.state)
        mx.clear_cache()
        logs.append({"step": step, "loss": float(loss_value.item())})
    return logs

optimizer = optim.AdamW(
    learning_rate=training_args["learning_rate"],
    weight_decay=training_args["weight_decay"],
)
loss_and_grad = nn.value_and_grad(
    model,
    lambda model, batch: _grpo_loss(model, batch, epsilon=training_args["epsilon"], epsilon_high=training_args["epsilon_high"]),
)

train_history = []


And let's train the model!

**NOTE** The default notebook settings now do a tiny notebook-local warm start before the RL updates so the fresh-start run produces a meaningful signal on a laptop. It still starts from the base Gemma 4 model and still performs RL afterward.


In [ ]:
bootstrap_logs = run_bootstrap_warm_start(
    model,
    steps=training_args["bootstrap_steps"],
    learning_rate=training_args["bootstrap_learning_rate"],
)
print('bootstrap_last_loss =', bootstrap_logs[-1] if bootstrap_logs else None)

warm_start_text, _, _ = sample_strategy(
    model,
    tokenizer,
    prompt_tokens,
    response_prefix=response_prefix,
    temperature=0.0,
    top_p=1.0,
    top_k=0,
    max_new_tokens=FINAL_SAMPLE_MAX_NEW_TOKENS,
    sampler_seed=4321,
)
selection_seeds = NOTEBOOK_HOLDOUT_SEEDS[:4]
best_mean_reward = evaluate_policy_mean_reward(warm_start_text, selection_seeds)
best_trainable_parameters = _clone_trainable_parameters(model)
print('warm_start_mean_holdout_reward =', best_mean_reward)

for step in range(1, training_args["max_steps"] + 1):
    rollout_batch = collect_rollout_samples(
        model,
        tokenizer,
        prompt_tokens,
        response_prefix=response_prefix,
        step=step,
        num_generations=training_args["num_generations"],
        max_new_tokens=training_args["sample_max_new_tokens"],
    )

    completions = [[{"content": item["completion_text"]}] for item in rollout_batch]
    completion_tokens_batch = [item["completion_tokens"] for item in rollout_batch]
    sampled_logprobs_batch = [item["sampled_logprobs"] for item in rollout_batch]

    print(f"=== training step {step} ===")
    print('unique_rollouts =', len({item["fingerprint"] for item in rollout_batch}))
    rollout_seeds = [NOTEBOOK_DIFFICULTY * 10000 + step * 100 + i for i in range(len(rollout_batch))]

    reward_1 = function_works(completions)
    reward_2 = no_cheating(completions)
    reward_3 = strategy_succeeds(completions, seeds=rollout_seeds)
    total_rewards = [a + b + c for a, b, c in zip(reward_1, reward_2, reward_3)]
    advantages = _normalize_rewards(total_rewards)

    rollout_samples = [
        type(
            "NotebookRolloutSample",
            (),
            {
                "completion_text": completions[i][0]["content"],
                "completion_tokens": completion_tokens_batch[i],
                "sampled_logprobs": sampled_logprobs_batch[i],
            },
        )()
        for i in range(len(completions))
    ]

    for i, completion in enumerate(completions):
        item = rollout_batch[i]
        print(
            'sample',
            i,
            'seed =', rollout_seeds[i],
            'temp =', item['temperature'],
            'top_p =', item['top_p'],
            'top_k =', item['top_k'],
            'reward =', total_rewards[i],
        )
        print(completion[0]["content"][:300])
        print('---')

    batch = _build_training_batch(prompt_tokens, rollout_samples, advantages)
    loss_value, grad = loss_and_grad(model, batch)
    optimizer.update(model, grad)
    mx.eval(model.state, optimizer.state)
    mx.clear_cache()

    post_train_text, _, _ = sample_strategy(
        model,
        tokenizer,
        prompt_tokens,
        response_prefix=response_prefix,
        temperature=0.0,
        top_p=1.0,
        top_k=0,
        max_new_tokens=FINAL_SAMPLE_MAX_NEW_TOKENS,
        sampler_seed=9000 + step,
    )
    post_train_mean_reward = evaluate_policy_mean_reward(post_train_text, selection_seeds)
    if post_train_mean_reward >= best_mean_reward:
        best_mean_reward = post_train_mean_reward
        best_trainable_parameters = _clone_trainable_parameters(model)

    summary = {
        "step": step,
        "loss": float(loss_value.item()),
        "reward_1": reward_1,
        "reward_2": reward_2,
        "reward_3": reward_3,
        "total_rewards": total_rewards,
        "advantages": advantages,
        "post_train_mean_holdout_reward": post_train_mean_reward,
        "best_mean_holdout_reward": best_mean_reward,
    }
    train_history.append(summary)
    print(summary)

model.update(best_trainable_parameters)
mx.eval(model.state)
mx.clear_cache()
print('restored_best_mean_holdout_reward =', best_mean_reward)


And now with the LoRA we just trained with RL - we first save the LoRA adapter locally.


In [ ]:
output_dir = REPO_ROOT / training_args["output_dir"]
adapter_dir = output_dir / "adapters"
adapter_dir.mkdir(parents=True, exist_ok=True)
adapter_file = adapter_dir / "adapters.safetensors"
adapter_config_file = adapter_dir / "adapter_config.json"

mx.save_safetensors(str(adapter_file), dict(tree_flatten(model.trainable_parameters())))
adapter_config = {
    "model": str(local_model_path),
    "fine_tune_type": "lora",
    "num_layers": 8,
    "lora_parameters": {
        "rank": lora_rank,
        "dropout": 0.0,
        "scale": lora_rank * 2,
        "keys": sorted([
            "self_attn.q_proj",
            "self_attn.k_proj",
            "self_attn.v_proj",
            "self_attn.o_proj",
            "mlp.gate_proj",
            "mlp.up_proj",
            "mlp.down_proj",
        ]),
    },
    "step": training_args["max_steps"],
}
adapter_config_file.write_text(json.dumps(adapter_config, indent=2, sort_keys=True))
print("saved_adapter_dir =", adapter_dir)


Verify LoRA is actually trained!


In [ ]:
print("adapter_file_exists =", adapter_file.exists())
print("adapter_file_size =", adapter_file.stat().st_size if adapter_file.exists() else 0)
print("adapter_config =")
print(adapter_config_file.read_text())


<a name="Inference"></a>
# Inference
Now let's try the model we just trained!


In [ ]:
del model
mx.clear_cache()

reloaded_model, reloaded_tokenizer = load(
    str(local_model_path),
    tokenizer_config={"trust_remote_code": True},
    adapter_path=str(adapter_dir),
)

trained_completion_text, trained_completion_tokens, trained_logprobs = sample_strategy(
    reloaded_model,
    reloaded_tokenizer,
    prompt_tokens,
    response_prefix=response_prefix,
    temperature=0.0,
    top_p=1.0,
    top_k=0,
    max_new_tokens=FINAL_SAMPLE_MAX_NEW_TOKENS,
)
print(trained_completion_text)


<a name="Save"></a>
### Saving to local MLX adapters

The main portable artifact in this standalone notebook is the local LoRA adapter directory.


In [ ]:
print("Adapter directory:", adapter_dir)
print("Files:")
for path in sorted(adapter_dir.glob('*')):
    print(" ", path.name)


### Optional export note
This standalone notebook focuses on local MLX adapters. If you want GGUF or app-specific integration later, treat that as a separate post-training step.


In [ ]:
EXPORT_NOTE = False
if EXPORT_NOTE:
    print("Add your export flow here.")
else:
    print("This notebook stops at local MLX adapter save/load.")


Now, we can also score the freshly reloaded policy on a tiny holdout set to close the loop.


In [ ]:
holdout_scores = []
for seed in NOTEBOOK_HOLDOUT_SEEDS:
    components = score_completion_on_seed(trained_completion_text, seed)
    holdout_scores.append(components)
    print(f"seed {seed} reward components =", components)

mean_holdout_reward = sum(item['total_reward'] for item in holdout_scores) / len(holdout_scores)
mean_strategy_reward = sum(item['strategy_reward'] for item in holdout_scores) / len(holdout_scores)
solved_fraction = sum(1 for item in holdout_scores if item['strategy_reward'] >= 30.0) / len(holdout_scores)
print('mean_holdout_reward =', mean_holdout_reward)
print('mean_strategy_reward =', mean_strategy_reward)
print('solved_fraction =', solved_fraction)


And we're done. This notebook now mirrors the original teaching flow as closely as possible while staying fully standalone and local.
